# Language Technology - 3rd Tutorial on Word Embeddings (Word2Vec, FastText) using Gensim


----
## Important Resources

Language Technology Resources: https://eclass.aueb.gr/modules/document/index.php?course=INF210

Python Official Documentation: https://docs.python.org/3.5/

NLTK: https://www.nltk.org

Gensim: https://radimrehurek.com/gensim/models/word2vec.html

Word2vec: https://code.google.com/archive/p/word2vec/

----

## Tutorial 5 Schedule

**In order to familiarize ourselves with word embeddings, we have the following sections:**

* **Train Word Embeddings**
    * **Data Cleansing**
    * **Lazy Load Data**
    * **Train Word2Vec Model**
    * **Train FastText Model**
* **Use Word Embeddings**
* **Train Word Embeddings with extra special tokens**

----
# Background

By the term word embedding in natural language processing (NLP), we describe a feature representation where words (tokens) or phrases (multi-token) from the vocabulary are mapped to vectors of real numbers. Distributed representations of words in a vector space help learning algorithms to achieve better performance in natural language processing tasks by grouping similar words. The ultimate leverage of such a technique is the transition from the traditional sparse features (i.e. one-hot vector representation) onto the dense vector space of common shared features.


## Word2vec model sampling

_"You shall know a word by the company it keeps" (Firth, J. R. 1957:11)_

<img src="https://i.imgur.com/v34sAaT.png" width="700">




## Word2vec model architecture

<img src="https://lilianweng.github.io/lil-log/assets/images/word2vec-skip-gram.png" width="700">



## Publicly available word embeddings

There are various pre-trained word embeddings, trained on large corpora such as Google Vectors, which are trained over a vast corpus in Google News containing 100-billion words and finally producing a vocabulary of 3 million words.

See here (https://github.com/3Top/word2vec-api#where-to-get-a-pretrained-models) for a list of every popular pre-trained word embeddings set.


----
# Train Word Embeddings

In order to train word embeddings with gensim library, we have to prepare a list of sentences, which then will be fed to the word2vec model. We need to address this matter in a memory friendly way. In this tutotial we are going to follow a 2-step approach:


## Data Cleansing - Preprocessing

First we will clean (pre-process) our corpus by producing a new set of files in which every single line will be a sentence and sentence's words will be splitted by a simple space.

**Python Code Example**

In [ ]:
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk.corpus import gutenberg
import os
import re

PROCESSED_CORPUS_FOLDER = '/Users/kiddo/Desktop/word2vec_corpus'

# Create new folder if it does not exist
if not os.path.exists(PROCESSED_CORPUS_FOLDER):
    os.mkdir(PROCESSED_CORPUS_FOLDER)

# Get filenames for Gutenberg corpus
file_ids = gutenberg.fileids()

print(file_ids)
print('{} BOOKS LOADED!'.format(len(file_ids)))

# Iterate over filenames
for filename in file_ids:
    # Create new file including one sentence per line, tokenized on white-space 
    with open(os.path.join(PROCESSED_CORPUS_FOLDER, filename), 'w', encoding='utf-8') as output_file:
        input_text = gutenberg.raw(filename)
        for sentence in sent_tokenize(input_text):
            sentence = re.sub('\n+', ' ', sentence)
            splitted_sentence = ' '.join([token.lower() for token in word_tokenize(sentence)]) + '\n'
            output_file.write(splitted_sentence)
    print(filename + ' DONE')

----
## Lazy Load Data

Secondly we will create a generator, which will load the sentences in a lazy-loading fashion.

**Python Code Example**

In [ ]:
import os
import glob

class CorpusLoader(object):

    def __init__(self,parent_folder=None):
        self.dirname = parent_folder

    def __iter__(self):
        for fname in glob.glob(os.path.join(self.dirname, '*.txt')):
            for line in open(os.path.join(self.dirname, fname)):
                yield line.split()


## Train Word2Vec Model

Initialize word2vec model with specific parameters and train using the cleansed corpus and our generator.

**Python Code Example**

In [ ]:
from gensim.models import Word2Vec
from multiprocessing import cpu_count

# CREATE GENERATOR OBJECT TO STREAM SENTENCES
sentences = CorpusLoader(parent_folder=PROCESSED_CORPUS_FOLDER)

# CONFIGURE WORD2VEC MODEL
model = Word2Vec(min_count=10, workers=cpu_count(), size=50, sg=1, window=5)

# BUILD VOCABULARY FROM SENTENCES CONSIDERING MIN_COUNT
print('START BUILDING VOCABULARY...')
model.build_vocab(sentences)
print('VOCABULARY_SIZE: ',len(model.wv.index2word))

# TRAIN MODEL
print('START TRAINING...')
model.train(sentences, total_examples=model.corpus_count, epochs=10)

model.wv.save_word2vec_format(os.path.join(PROCESSED_CORPUS_FOLDER, 'WORD2VEC.bin'), binary=True)
print('MODEL SAVED....')

## Train FastText Model

*Python Code Example*

In [ ]:
from gensim.models import FastText

from multiprocessing import cpu_count

# CREATE GENERATOR OBJECT TO STREAM SENTENCES
sentences = CorpusLoader(parent_folder=PROCESSED_CORPUS_FOLDER)

# CONFIGURE WORD2VEC MODEL
model = FastText(workers=cpu_count(), size=50, sg=1, window=5)

# BUILD VOCABULARY FROM SENTENCES CONSIDERING MIN_COUNT
print('START BUILDING VOCABULARY...')
model.build_vocab(sentences)
print('VOCABULARY_SIZE: ',len(model.wv.index2word))

# TRAIN MODEL
print('START TRAINING...')
model.train(sentences, total_examples=model.corpus_count, epochs=10)

model.save(os.path.join(PROCESSED_CORPUS_FOLDER, 'FASTTEXT.bin'))
print('MODEL SAVED....')

----
# Use word embeddings

**Python Code Example**

In [ ]:
from gensim.models import KeyedVectors

# LOAD WORD2VEC LOOKUP TABLE
w2v_model = KeyedVectors.load_word2vec_format(os.path.join(PROCESSED_CORPUS_FOLDER, 'WORD2VEC.bin'), binary=True)

print(w2v_model['good'])

In [ ]:
# SEARCH (LOOKUP) WORD EMBEDDINGS
words = ['night', 'dog', 'umbrella', 'shoe', 'man', 'woman', 'boat', 'love', 'george', '5', '587']

# FIND MOST SIMILAR WORDS
for word in words:
    if word in w2v_model:
        record = '{:8}\n'.format(word)
        record += '--------------------\n'
        for sim_word, sim_score in w2v_model.most_similar(positive=[word]):
            record += '{:15}| {:.2f}  \n'.format(sim_word, sim_score)
        print(record)
    else:
        print('{:8} NOT FOUND'.format(word))


In [ ]:
# SIMILARITY MEASURE
print(w2v_model.similarity('man', 'woman'))
print(w2v_model.similarity('man', 'pig'))
print(w2v_model.similarity('man', 'chair'))

 FastText in contrast with Word2Vec does not rely on a fixed vocabulary, so we need to load the actual trained model.

In [ ]:
fasttext_model = FastText.load(os.path.join(PROCESSED_CORPUS_FOLDER, 'FASTTEXT.bin'))

# SEARCH FASTTEXT MODEL
words = ['night', 'dog', 'umbrella', 'shoe', 'man', 'woman', 'boat', 'love', 'george', '5', 'dog-walker']

# FIND MOST SIMILAR WORDS
for word in words:
    record = '{:8}\n'.format(word)
    record += '--------------------\n'
    for sim_word, sim_score in fasttext_model.wv.most_similar(positive=[word]):
        record += '{:15}| {:.2f}  \n'.format(sim_word, sim_score)
    print(record)

## Train Word2Vec Model with extra special tokens

(a) Normalize numbers, (b) Create unknown token, (c) Preserve newline character

**Python Code Example**

In [ ]:
from collections import Counter

# Create new folder if it does not exist
if not os.path.exists(PROCESSED_CORPUS_FOLDER):
    os.mkdir(PROCESSED_CORPUS_FOLDER)

# Get filenames for Gutenberg corpus
file_ids = gutenberg.fileids()

print(file_ids)
print('{} BOOKS LOADED!'.format(len(file_ids)))

tokens = []

# Iterate over filenames
for filename in file_ids:
    # Create new file including one sentence per line, tokenized on white-space 
    input_text = gutenberg.raw(filename)
    for sentence in sent_tokenize(input_text):
        # Preserve newline with snowmam :D
        sentence = re.sub('\n+', ' ?? ', sentence)
        tokens.extend([token.lower() if not re.search('[0-9]', token) else re.sub('[0-9]', 'D', token.lower())
                       for token in word_tokenize(sentence)])
print('TOTAL TOKENS IN CORPUS: {}'.format(len(tokens)))
vocab = Counter(tokens)

print('10 MOST COMMON TOKENS: {}'.format(vocab.most_common(10)))

# ELIMINATE RARE WORDS (<10)
vocab = {k:v for k,v in vocab.most_common() if v >= 10}

# Iterate over filenames
for filename in file_ids:
    # Create new file including one sentence per line, tokenized on white-space 
    with open(os.path.join(PROCESSED_CORPUS_FOLDER, filename), 'w', encoding='utf-8') as output_file:
        input_text = gutenberg.raw(filename)
        for sentence in sent_tokenize(input_text):
            sentence = re.sub('\n+', ' ?? ', sentence)
            normalized_tokens = []
            for token in word_tokenize(sentence):
                if token == '??':
                    normalized_tokens.append('#NEWLINE#')
                elif re.search('[0-9]', token):
                    token_norm = re.sub('[0-9]', 'D', token.lower())
                    if token_norm in vocab:
                        normalized_tokens.append(token_norm)
                    else:
                        normalized_tokens.append('#UNK#')
                else:
                    if token.lower() in vocab:
                        normalized_tokens.append(token.lower())
                    else:
                        normalized_tokens.append('#UNK#')
            splitted_sentence = ' '.join(normalized_tokens) + '\n'
            output_file.write(splitted_sentence)
    print(filename + ' DONE')
    

from gensim.models import Word2Vec
from multiprocessing import cpu_count

# CREATE GENERATOR OBJECT TO STREAM SENTENCES
sentences = CorpusLoader(parent_folder=PROCESSED_CORPUS_FOLDER)

# CONFIGURE WORD2VEC MODEL
model = Word2Vec(workers=cpu_count(), size=50, sg=1, window=5)

# BUILD VOCABULARY FROM SENTENCES CONSIDERING MIN_COUNT
print('START BUILDING VOCABULARY...')
model.build_vocab(sentences)
print('VOCABULARY_SIZE: ',len(model.wv.index2word))

# TRAIN MODEL
print('START TRAINING...')
model.train(sentences, total_examples=model.corpus_count, epochs=10)

model.wv.save_word2vec_format(os.path.join(PROCESSED_CORPUS_FOLDER, 'WORD2VEC_UNK.bin'), binary=True)
print('MODEL SAVED....')

In [ ]:
from gensim.models import KeyedVectors

# LOAD WORD2VEC LOOKUP TABLE
w2v_model = KeyedVectors.load_word2vec_format(os.path.join(PROCESSED_CORPUS_FOLDER, 'WORD2VEC_UNK.bin'), binary=True)

special_tokens = ['#UNK#', '#NEWLINE#', 'DD', 'DDDD', 'D,DDD']

# FIND MOST SIMILAR WORDS
for word in special_tokens:
    if word in w2v_model:
        record = '{:8}\n'.format(word)
        record += '--------------------\n'
        for sim_word, sim_score in w2v_model.most_similar(positive=[word]):
            record += '{:15}| {:.2f}  \n'.format(sim_word, sim_score)
        print(record)
    else:
        print('{:8} NOT FOUND'.format(word))